<a href="https://colab.research.google.com/github/sreent/data-management-intro/blob/main/Document%20Modelling/06-lab-document-modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Topic 6 Lab — Document Modelling: JSON, MongoDB, and Aggregate Design

This lab builds two MongoDB designs for the **super-app personalisation layer** and compares them against a shared workload. You construct and insert your own documents in Parts 1 and 2, then query, compare, and analyse both designs in Parts 3–6. The focus is on aggregate design, embed-or-reference decisions, query patterns, and the trade-offs between aggregate-first and reference-heavy approaches.

Use the formal vocabulary (document, collection, aggregate, embed, reference, snapshot, bounded growth, schema-on-read, `$lookup`, `$unwind`, `$elemMatch`, dot notation, selectivity) throughout your work.

**Time budget:** Part 1 ~25 min | Part 2 ~20 min | Part 3 ~20 min | Part 4 ~25 min | Part 5 ~15 min | Part 6 ~15 min | **Total ~120 min**

**Tools required:** Google Colab with MongoDB (`%%mongodb` magic via `cellspell`)

---

## Setup

Run the following cells in a new Google Colab notebook. Unlike Chapters 2–5, where base schemas were pre-loaded, you construct and insert your own documents in Parts 1 and 2. A seed data cell in Part 2 loads additional data for Parts 3–6.

**Cell 1 — Install MongoDB:**

In [ ]:
# Topic 6 — Document Modelling: Super-app Personalisation Lab
!wget -q http://archive.ubuntu.com/ubuntu/pool/main/o/openssl/libssl1.1_1.1.1f-1ubuntu2_amd64.deb
!dpkg -i libssl1.1_1.1.1f-1ubuntu2_amd64.deb > /dev/null 2>&1
!wget -qO - https://www.mongodb.org/static/pgp/server-4.4.asc | apt-key add - > /dev/null 2>&1
!echo "deb [ arch=amd64,arm64 ] http://repo.mongodb.org/apt/ubuntu bionic/mongodb-org/4.4 multiverse" | tee /etc/apt/sources.list.d/mongodb-org-4.4.list > /dev/null
!apt-get update -qq > /dev/null
!apt-get install -y -qq mongodb-org > /dev/null
!mkdir -p /data/db
!mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
!mongo --quiet --eval 'print("MongoDB ready!")'


**Cell 2 — Install cellspell:**

In [ ]:
!pip install "cellspell[mongodb] @ git+https://github.com/sreent/jupyter-query-magics.git" -q
%load_ext cellspell.mongodb


**Cell 3 — Connect:**

In [ ]:
%mongodb mongodb://localhost:27017/week6


**Cell 3b — Reset database (safe re-run):**


In [ ]:
%%mongodb
db.dropDatabase()


**Cell 4 — Verify empty database:**

In [ ]:
%%mongodb
db.users.countDocuments({})



> **Expected: 0 — empty database ready for student work**



---

## Part 1 — Design A: Aggregate-First (Embedded)

### Q1. Document shape — write and insert

Design the `users` collection with role-specific document shapes. Write `insertMany` to insert 7 representative user documents — three passengers, two drivers, and two merchants — plus 4 plan documents into a `plans` collection.

This is the core exercise: constructing the JSON *is* the design. The documents you write should look like what the API returns — if they do not, the shape is wrong.

**(a)** Insert user documents:

In [ ]:
%%mongodb
db.users.insertMany([
    {
        "userId": "passenger-1001",
        "displayName": "Sarah Tan",
        "phone": "+6591234567",
        "email": "sarah@example.com",
        "role": "passenger",
        "tier": "standard",
        "savedPlaces": [
            {"label": "Home", "address": "123 Clementi Ave 3", "lat": 1.315, "lng": 103.765, "frequency": "daily", "lastUsed": "2025-01-20"},
            {"label": "Office", "address": "1 Raffles Place", "lat": 1.284, "lng": 103.851, "frequency": "weekday", "lastUsed": "2024-12-10"}
        ],
        "paymentMethods": [
            {"type": "card", "last4": "4242", "isDefault": true},
            {"type": "grabpay", "balance": 45.00}
        ],
        "ridePreferences": {"vehicleType": "JustGrab", "quietRide": false, "temperature": "normal"},
        "rewardPoints": 2840
    },
    {
        "userId": "passenger-1002",
        "displayName": "Wei Ming Lim",
        "phone": "+6598761234",
        "email": "weiming@example.com",
        "role": "passenger",
        "tier": "premium",
        "savedPlaces": [
            {"label": "Home", "address": "88 Bukit Timah Rd", "lat": 1.328, "lng": 103.797, "frequency": "daily", "lastUsed": "2025-01-18"},
            {"label": "Gym", "address": "10 Bayfront Ave", "lat": 1.283, "lng": 103.860, "frequency": "weekly", "lastUsed": "2025-01-15"},
            {"label": "Parents", "address": "45 Ang Mo Kio Ave 6", "lat": 1.370, "lng": 103.837, "frequency": "monthly", "lastUsed": "2024-12-25"}
        ],
        "paymentMethods": [
            {"type": "card", "last4": "8888", "isDefault": true}
        ],
        "ridePreferences": {"vehicleType": "GrabCar Premium", "quietRide": true, "temperature": "cold"},
        "loyaltyTier": "platinum",
        "loungeAccess": true,
        "rewardPoints": 15200,
        "subscriptionSnapshot": {
            "planNameAtSignUp": "GrabCar Premium",
            "priceAtSignUp": 29.99,
            "signUpDate": "2023-06-01"
        }
    },
    {
        "userId": "passenger-1003",
        "displayName": "Rani Devi",
        "phone": "+6587001234",
        "email": "rani@example.com",
        "role": "passenger",
        "tier": "standard",
        "savedPlaces": [],
        "paymentMethods": [
            {"type": "grabpay", "balance": 120.00}
        ],
        "ridePreferences": {"vehicleType": "GrabShare", "quietRide": false, "temperature": "normal"},
        "rewardPoints": 450
    },
    {
        "userId": "driver-5582",
        "displayName": "Ahmad Ibrahim",
        "phone": "+6598765432",
        "email": "ahmad@example.com",
        "role": "driver",
        "driverType": "grabcar",
        "vehicle": {
            "make": "Toyota",
            "model": "Prius",
            "year": 2022,
            "plateNumber": "SBA1234X",
            "vehicleType": "standard"
        },
        "license": {
            "licenseNumber": "S1234567A",
            "expiryDate": "2026-08-15",
            "vocationalLicenseExpiry": "2025-12-31"
        },
        "preferences": {
            "maxDistance": 25,
            "preferredZones": ["central", "east"],
            "autoAccept": false
        },
        "earnings": {
            "currentWeek": 1240.50,
            "incentiveProgress": {
                "ridesCompleted": 42,
                "target": 50,
                "bonus": 120.00
            }
        },
        "subscriptionSnapshot": {
            "planNameAtSignUp": "GrabCar Premium",
            "commissionRateAtSignUp": 0.20,
            "signUpDate": "2024-01-15"
        }
    },
    {
        "userId": "driver-5583",
        "displayName": "Priya Nair",
        "phone": "+6587654321",
        "email": "priya@example.com",
        "role": "driver",
        "driverType": "grabfood",
        "bike": {
            "type": "electric_scooter",
            "model": "Niu NQi GTS",
            "plateNumber": "FBA5678Y"
        },
        "thermalBagCertified": true,
        "deliveryZone": "central",
        "license": {
            "licenseNumber": "S7654321B",
            "expiryDate": "2025-06-30",
            "vocationalLicenseExpiry": "2025-06-30"
        },
        "preferences": {
            "maxDistance": 10,
            "preferredZones": ["central"],
            "autoAccept": true
        },
        "earnings": {
            "currentWeek": 680.00,
            "incentiveProgress": {
                "deliveriesCompleted": 95,
                "target": 100,
                "bonus": 50.00
            }
        },
        "subscriptionSnapshot": {
            "planNameAtSignUp": "GrabFood Partner",
            "commissionRateAtSignUp": 0.30,
            "signUpDate": "2024-03-10"
        }
    },
    {
        "userId": "merchant-0891",
        "displayName": "Ah Huat",
        "phone": "+6562345678",
        "email": "ahhuat@example.com",
        "role": "merchant",
        "businessName": "Ah Huat Chicken Rice",
        "cuisine": "Chinese",
        "operatingHours": {
            "monday": {"open": "10:00", "close": "21:00"},
            "tuesday": {"open": "10:00", "close": "21:00"},
            "wednesday": {"open": "10:00", "close": "21:00"},
            "thursday": {"open": "10:00", "close": "21:00"},
            "friday": {"open": "10:00", "close": "22:00"},
            "saturday": {"open": "09:00", "close": "22:00"},
            "sunday": null
        },
        "menu": [
            {"itemName": "Roasted Chicken Rice", "price": 5.50, "category": "mains",
             "customisations": [{"name": "Extra rice", "price": 0.50}, {"name": "Add egg", "price": 1.00}]},
            {"itemName": "Steamed Chicken Rice", "price": 5.00, "category": "mains",
             "customisations": [{"name": "Extra rice", "price": 0.50}]},
            {"itemName": "Char Siew Rice", "price": 6.00, "category": "mains",
             "customisations": [{"name": "Extra rice", "price": 0.50}]},
            {"itemName": "Iced Barley", "price": 1.50, "category": "drinks",
             "customisations": []},
            {"itemName": "Hot Tea", "price": 1.20, "category": "drinks",
             "customisations": []}
        ],
        "promotions": [
            {"code": "AHHUAT10", "discount": 0.10, "validUntil": "2025-03-31"}
        ],
        "subscriptionSnapshot": {
            "planNameAtSignUp": "GrabFood Partner",
            "commissionRateAtSignUp": 0.30,
            "signUpDate": "2023-11-01"
        }
    },
    {
        "userId": "merchant-0892",
        "displayName": "Jia Yi",
        "phone": "+6563456789",
        "email": "jiayi@example.com",
        "role": "merchant",
        "businessName": "Tiger Sugar Toa Payoh",
        "cuisine": "Beverages",
        "operatingHours": {
            "monday": {"open": "11:00", "close": "22:00"},
            "tuesday": {"open": "11:00", "close": "22:00"},
            "wednesday": {"open": "11:00", "close": "22:00"},
            "thursday": {"open": "11:00", "close": "22:00"},
            "friday": {"open": "11:00", "close": "23:00"},
            "saturday": {"open": "10:00", "close": "23:00"},
            "sunday": {"open": "10:00", "close": "21:00"}
        },
        "menu": [
            {"itemName": "Brown Sugar Boba Milk", "price": 6.90, "category": "signature",
             "customisations": [{"name": "Less sugar", "price": 0.00}, {"name": "Extra boba", "price": 1.00}]},
            {"itemName": "Tiger Black Sugar", "price": 5.90, "category": "signature",
             "customisations": [{"name": "Less sugar", "price": 0.00}]},
            {"itemName": "Oolong Tea Latte", "price": 5.50, "category": "tea",
             "customisations": [{"name": "Less sugar", "price": 0.00}, {"name": "Oat milk", "price": 0.80}]},
            {"itemName": "Matcha Latte", "price": 6.50, "category": "tea",
             "customisations": [{"name": "Less sugar", "price": 0.00}]}
        ],
        "promotions": [],
        "subscriptionSnapshot": {
            "planNameAtSignUp": "GrabFood Standard",
            "commissionRateAtSignUp": 0.35,
            "signUpDate": "2024-02-15"
        }
    }
])


**(b)** Insert plan documents:

In [ ]:
%%mongodb
db.plans.insertMany([
    {"planId": "plan-premium", "name": "GrabCar Premium", "commissionRate": 0.20, "features": ["priority-matching", "lounge-access", "premium-support"], "monthlyPrice": 29.99, "tier": "premium"},
    {"planId": "plan-standard", "name": "GrabCar Standard", "commissionRate": 0.25, "features": ["standard-matching"], "monthlyPrice": 0.00, "tier": "standard"},
    {"planId": "plan-grabfood-partner", "name": "GrabFood Partner", "commissionRate": 0.30, "features": ["food-delivery", "merchant-dashboard"], "monthlyPrice": 0.00, "tier": "partner"},
    {"planId": "plan-grabfood-standard", "name": "GrabFood Standard", "commissionRate": 0.35, "features": ["food-delivery"], "monthlyPrice": 0.00, "tier": "standard"}
])


**(c)** Verify your inserts:

In [ ]:
%%mongodb
db.users.countDocuments({})



> **Expected: 7**


In [ ]:
%%mongodb
db.plans.countDocuments({})



> **Expected: 4**



---

### Q2. Bounded-growth rules

For each embedded array in your Design A documents, state: (a) the upper bound, (b) the business rule that enforces it, (c) the fallback if the bound is exceeded.

Complete the table:

| Array | Upper bound | Business rule | Fallback |
|---|---|---|---|
| `savedPlaces[]` | max 10 | App UI disables add-place button at limit | User must remove one before adding another |
| `paymentMethods[]` | ? | ? | ? |
| `menu[]` | ? | ? | ? |
| `menu[].customisations[]` | ? | ? | ? |
| `promotions[]` | ? | ? | ? |
| `preferences.preferredZones[]` | ? | ? | ? |

Also explain: why is `rideHistory` not embedded? Which condition of the embed-or-reference test applies?

---

### Q3. Schema validation

The `users` collection currently accepts any JSON — schema-on-read. Add a validation rule that rejects documents with: (a) missing `userId`, (b) missing `role`, (c) `role` value not in the allowed set.

**(a)** Create a validated collection:

In [ ]:
%%mongodb
db.createCollection("users_validated", {
    validator: {
        $jsonSchema: {
            bsonType: "object",
            required: ["userId", "role"],
            properties: {
                userId: { bsonType: "string" },
                role: { enum: ["passenger", "driver", "merchant"] }
            }
        }
    }
})


**(b)** *Predict before running:* What happens when you insert a document with no `userId` and no `role`?

In [ ]:
%%mongodb
db.users_validated.insertOne({"displayName": "No userId or role"})



> **Predict: accepted or rejected?**



**(c)** Test with invalid role:

In [ ]:
%%mongodb
db.users_validated.insertOne({"userId": "test-bad", "role": "admin"})



> **Predict: accepted or rejected?**



**(d)** Test with valid document:

In [ ]:
%%mongodb
db.users_validated.insertOne({"userId": "test-001", "role": "passenger", "displayName": "Valid User"})



> **Predict: accepted or rejected?**



**(e)** *Discussion:* The corporate admin document from sanity check 2 (`role: "corporate_admin"`) would be rejected by this validation rule. What does this tell you about the trade-off between schema-on-read flexibility and schema-on-write enforcement?

---

## Part 2 — Design B: Reference-Heavy (Microservices Decomposition)

### Q4. Collection structure — write and insert

Design the same domain with separate collections simulating a microservices decomposition. Each collection stores one concern; references link them via `userId`.

**(a)** Core identity:

In [ ]:
%%mongodb
db.users_b.insertMany([
    {"userId": "passenger-1001", "displayName": "Sarah Tan", "phone": "+6591234567", "email": "sarah@example.com", "role": "passenger", "tier": "standard"},
    {"userId": "passenger-1002", "displayName": "Wei Ming Lim", "phone": "+6598761234", "email": "weiming@example.com", "role": "passenger", "tier": "premium"},
    {"userId": "passenger-1003", "displayName": "Rani Devi", "phone": "+6587001234", "email": "rani@example.com", "role": "passenger", "tier": "standard"},
    {"userId": "driver-5582", "displayName": "Ahmad Ibrahim", "phone": "+6598765432", "email": "ahmad@example.com", "role": "driver", "driverType": "grabcar"},
    {"userId": "driver-5583", "displayName": "Priya Nair", "phone": "+6587654321", "email": "priya@example.com", "role": "driver", "driverType": "grabfood"},
    {"userId": "merchant-0891", "displayName": "Ah Huat", "phone": "+6562345678", "email": "ahhuat@example.com", "role": "merchant", "businessName": "Ah Huat Chicken Rice"},
    {"userId": "merchant-0892", "displayName": "Jia Yi", "phone": "+6563456789", "email": "jiayi@example.com", "role": "merchant", "businessName": "Tiger Sugar Toa Payoh"}
])


**(b)** Saved places (each document references a userId):

In [ ]:
%%mongodb
db.savedPlaces_b.insertMany([
    {"userId": "passenger-1001", "label": "Home", "address": "123 Clementi Ave 3", "lat": 1.315, "lng": 103.765, "frequency": "daily", "lastUsed": "2025-01-20"},
    {"userId": "passenger-1001", "label": "Office", "address": "1 Raffles Place", "lat": 1.284, "lng": 103.851, "frequency": "weekday", "lastUsed": "2024-12-10"},
    {"userId": "passenger-1002", "label": "Home", "address": "88 Bukit Timah Rd", "lat": 1.328, "lng": 103.797, "frequency": "daily", "lastUsed": "2025-01-18"},
    {"userId": "passenger-1002", "label": "Gym", "address": "10 Bayfront Ave", "lat": 1.283, "lng": 103.860, "frequency": "weekly", "lastUsed": "2025-01-15"},
    {"userId": "passenger-1002", "label": "Parents", "address": "45 Ang Mo Kio Ave 6", "lat": 1.370, "lng": 103.837, "frequency": "monthly", "lastUsed": "2024-12-25"}
])


**(c)** Write similar `insertMany` cells for the remaining collections. Insert documents for the same users from Q1:

- `preferences_b` — ride preferences for passengers, preferred zones for drivers
- `vehicles_b` — vehicle details and licence information for drivers
- `earnings_b` — driver earnings and incentive progress
- `paymentMethods_b` — payment methods for all users who have them
- `menuItems_b` — each menu item as a separate document, referencing `merchantId`

**(d)** Verify counts match Design A's data:

In [ ]:
%%mongodb
db.users_b.countDocuments({})



> **Expected: 7**



---

### Q5. Reference resolution

**(a)** To fetch a complete driver profile in Design B (identity + vehicle + licence + preferences + earnings), how many queries or `$lookup` stages are required? List them.

**(b)** Compare to Design A's single `findOne`. Which returns the nested JSON shape directly?

**Q5 follow-up (Relational comparison):**

**(c)** Sketch the relational schema for a driver profile. Which tables would you need? (At minimum: `UserProfile`, a `DriverProfile` subtype table, a `PreferredZones` junction table.) How many JOINs for the profile-load query?

**(d)** Complete this comparison:

| Design | Collections/Tables | Reads for profile load |
|---|---|---|
| Design A (embedded) | 1 collection (`users`) | 1 `findOne` |
| Design B (referenced) | ? collections | ? queries or `$lookup` stages |
| Relational | ? tables | ? JOINs |

---

### Seed Data Cell

**Run this cell after completing Q4.** It downloads and executes a pre-generated dataset (~200 user profiles and ~10 plans) across both designs. The data is generated deterministically (seed 42) by `resources/ridelink/generate-lab-data.py`.

Do not modify this cell.

In [ ]:
# ── Seed data cell — pre-written, do not modify ──
# Downloads lab-data.js from the course repository and loads it into MongoDB.
import subprocess, urllib.request, os

LAB_DATA_URL = "https://raw.githubusercontent.com/sreent/data-management-intro/main/resources/ridelink/lab-data.js"
LAB_DATA_PATH = "/tmp/lab-data.js"

if not os.path.exists(LAB_DATA_PATH):
    urllib.request.urlretrieve(LAB_DATA_URL, LAB_DATA_PATH)
    print(f"Downloaded lab-data.js ({os.path.getsize(LAB_DATA_PATH):,} bytes)")

result = subprocess.run(
    ["mongo", "mongodb://localhost:27017/week6", LAB_DATA_PATH],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("Seed data loaded successfully.")
else:
    print(f"Error: {result.stderr[:500]}")


**Verify counts:**

In [ ]:
%%mongodb
db.users.countDocuments({})



> **Expected: ~207 (7 from Q1 + 200 from seed)**


In [ ]:
%%mongodb
db.users.countDocuments({"role": "passenger"})



> **Expected: ~123**


In [ ]:
%%mongodb
db.users.countDocuments({"role": "driver"})



> **Expected: ~52**


In [ ]:
%%mongodb
db.users.countDocuments({"role": "merchant"})



> **Expected: ~32**


In [ ]:
%%mongodb
db.plans.countDocuments({})



> **Expected: ~10**



---

## Part 3 — Query Patterns on Design A

These exercises practise the query techniques from the chapter — dot notation, `$elemMatch`, and regex — against the embedded Design A.

### Q6. Dot notation — nested field query

Find all drivers whose licence expires before 2026-01-01.

**(a)** *Predict before running:* How many drivers should match? (Consider: the seed data generates expiry dates in 2025, 2026, and 2027 — roughly a third in each year.)

**(b)** Run the query:

In [ ]:
%%mongodb
db.users.find(
    {"role": "driver", "license.expiryDate": {"$lt": "2026-01-01"}},
    {"userId": 1, "displayName": 1, "license.expiryDate": 1, "_id": 0}
)


**(c)** Compare your prediction to the actual count. Note: dates are stored as ISO-8601 strings, which sort lexicographically in correct chronological order.

---

### Q7a. `$elemMatch` — passenger saved places

Find passengers where a *single* saved place has frequency `"daily"` AND `lastUsed` before `"2025-01-15"`.

**(a)** Write the query *without* `$elemMatch`:

In [ ]:
%%mongodb
db.users.countDocuments(
    {"role": "passenger",
     "savedPlaces.frequency": "daily",
     "savedPlaces.lastUsed": {"$lt": "2025-01-15"}})


**(b)** Write the query *with* `$elemMatch`:

In [ ]:
%%mongodb
db.users.countDocuments(
    {"role": "passenger",
     "savedPlaces": {"$elemMatch": {"frequency": "daily", "lastUsed": {"$lt": "2025-01-15"}}}})


**(c)** *Predict before running:* Will the two queries return the same count? Why or why not?

**(d)** Run both and compare. Explain the difference: the without-`$elemMatch` version matches documents where *any* place is "daily" and *any* place (possibly different) has old `lastUsed`. The with-`$elemMatch` version matches only if a *single* place satisfies both.

---

### Q7b. `$elemMatch` — merchant menu items

Find merchants where a *single* menu item has category `"mains"` AND price less than 5.00.

**(a)** Without `$elemMatch`:

In [ ]:
%%mongodb
db.users.countDocuments(
    {"role": "merchant",
     "menu.category": "mains",
     "menu.price": {"$lt": 5.00}})


**(b)** With `$elemMatch`:

In [ ]:
%%mongodb
db.users.countDocuments(
    {"role": "merchant",
     "menu": {"$elemMatch": {"category": "mains", "price": {"$lt": 5.00}}}})


**(c)** Explain why the results may differ. Construct a scenario: a merchant has one menu item with category "mains" at price 8.00 and another with category "drinks" at price 3.50. The without-`$elemMatch` query matches (some item is "mains", some item is under 5.00); the with-`$elemMatch` query does not (no single item is both "mains" and under 5.00).

---

### Q8. Regex — pattern matching

**(a)** Find users whose phone number starts with `+65` (Singapore numbers):

In [ ]:
%%mongodb
db.users.countDocuments({"phone": /^\+65/})


> **Predict: how many? (All users have Singapore numbers in this dataset.)**



**(b)** Find merchants whose business name contains "chicken rice" (case-insensitive):

In [ ]:
%%mongodb
db.users.find({"role": "merchant", "businessName": /chicken rice/i},
              {"userId": 1, "businessName": 1, "_id": 0})


**(c)** Which query is index-friendly and which requires a collection scan? Why? (Hint: anchored `^` patterns can use a B-tree index; unanchored patterns cannot.)

---

## Part 4 — Same Workload on Both Designs

For each query below, implement it against *both* Design A and Design B. Record the query/pipeline for each and note the differences in complexity.

### Q9. Profile load — single aggregate read

Fetch a complete driver profile by `userId`.

**(a)** Design A:

In [ ]:
%%mongodb
db.users.findOne({"userId": "driver-5582"})


**(b)** Design B — write a `$lookup` pipeline that assembles the same result:

In [ ]:
%%mongodb
db.users_b.aggregate([
    {"$match": {"userId": "driver-5582"}},
    {"$lookup": {"from": "vehicles_b", "localField": "userId", "foreignField": "userId", "as": "vehicleInfo"}},
    {"$lookup": {"from": "preferences_b", "localField": "userId", "foreignField": "userId", "as": "preferencesInfo"}},
    {"$lookup": {"from": "earnings_b", "localField": "userId", "foreignField": "userId", "as": "earningsInfo"}},
    {"$lookup": {"from": "paymentMethods_b", "localField": "userId", "foreignField": "userId", "as": "paymentMethods"}}
])


**(c)** Record: how many queries or pipeline stages for each? Which returns the nested JSON shape directly?

---

### Q10. Nested field query — drivers with expiring licences

Find all drivers whose licence expires before 2026-06-01.

**(a)** Design A:

In [ ]:
%%mongodb
db.users.countDocuments(
    {"role": "driver", "license.expiryDate": {"$lt": "2026-06-01"}})


**(b)** Design B — query `vehicles_b` then resolve identity:

In [ ]:
%%mongodb
db.vehicles_b.aggregate([
    {"$match": {"license.expiryDate": {"$lt": "2026-06-01"}}},
    {"$lookup": {"from": "users_b", "localField": "userId", "foreignField": "userId", "as": "user"}},
    {"$unwind": "$user"},
    {"$project": {"userId": 1, "displayName": "$user.displayName", "expiryDate": "$license.expiryDate", "_id": 0}}
])


**(c)** Record: how many steps for each?

---

### Q11. Analytics — aggregation pipeline

"Top 5 food categories by item count and average price."

**(a)** Design A:

In [ ]:
%%mongodb
db.users.aggregate([
    {"$match": {"role": "merchant"}},
    {"$unwind": "$menu"},
    {"$group": {
        "_id": "$menu.category",
        "itemCount": {"$sum": 1},
        "avgPrice": {"$avg": "$menu.price"}
    }},
    {"$sort": {"itemCount": -1}},
    {"$limit": 5}
])


**(b)** *Predict before running:* Design A requires `$unwind` — why? (Because menu items are embedded arrays that must be deconstructed for grouping.)

**(c)** Design B:

In [ ]:
%%mongodb
db.menuItems_b.aggregate([
    {"$group": {
        "_id": "$category",
        "itemCount": {"$sum": 1},
        "avgPrice": {"$avg": "$price"}
    }},
    {"$sort": {"itemCount": -1}},
    {"$limit": 5}
])


**(d)** Record: how many pipeline stages for each? Which design requires `$unwind` and why?

**Q11 follow-up — Analytics across the aggregate boundary:**

**(e)** Compute "average menu item price by merchant subscription plan tier." Plan tier lives in the `plans` collection, not in the merchant profile. Extend the Design A pipeline:

In [ ]:
%%mongodb
db.users.aggregate([
    {"$match": {"role": "merchant"}},
    {"$lookup": {
        "from": "plans",
        "localField": "subscriptionSnapshot.planNameAtSignUp",
        "foreignField": "name",
        "as": "planInfo"
    }},
    {"$unwind": "$planInfo"},
    {"$unwind": "$menu"},
    {"$group": {
        "_id": "$planInfo.tier",
        "avgPrice": {"$avg": "$menu.price"},
        "itemCount": {"$sum": 1}
    }},
    {"$sort": {"avgPrice": -1}}
])


**(f)** How many stages does this pipeline have? Compare to the within-aggregate version above. This is the cost of crossing the aggregate boundary.

---

### Q12. Update complexity — merchant renames business

**(a)** Design A:

In [ ]:
%%mongodb
db.users.updateOne(
    {"userId": "merchant-0891"},
    {"$set": {"businessName": "Ah Huat Kitchen"}}
)


**(b)** Design B:

In [ ]:
%%mongodb
db.users_b.updateOne(
    {"userId": "merchant-0891"},
    {"$set": {"businessName": "Ah Huat Kitchen"}}
)


**(c)** For this particular field, both designs require one document update. Why?

**Q12 follow-up — Snapshot discipline:**

**(d)** The company changes the "GrabCar Premium" plan's commission rate from 20% to 18%. For each design, answer:

- What collections need updating?
- How many documents are affected?
- Are any user profile documents modified?

**(e)** Write the `updateOne` statement that applies the change. Then state: how many user profile documents are modified in Design A? In Design B? Why?

**(f)** *Discussion:* After the update, a driver who signed up under the old 20% rate opens the app. What commission rate does Design A show? What does Design B show? Which behaviour does the business need for commission disputes?

---

## Part 5 — Indexes and Explain

### Q13. Index creation and explain — before/after

**(a)** Run `explain("executionStats")` on Q10's Design A query *before* adding any index:

In [ ]:
%%mongodb
db.users.find({"role": "driver", "license.expiryDate": {"$lt": "2026-06-01"}}).explain("executionStats")


**(b)** Record: does the plan show COLLSCAN (collection scan)? How many documents were examined?

**(c)** Create a compound index:

In [ ]:
%%mongodb
db.users.createIndex({"role": 1, "license.expiryDate": 1})


**(d)** Re-run the explain:

In [ ]:
%%mongodb
db.users.find({"role": "driver", "license.expiryDate": {"$lt": "2026-06-01"}}).explain("executionStats")


**(e)** Record: did the plan change from COLLSCAN to IXSCAN? How many documents were examined? Why index `role` first, then `license.expiryDate`? (The compound index filters by role first — narrowing to ~25% of documents — then scans within that subset by date.)

---

### Q14. Selectivity prediction

**(a)** *Predict before running:* Which provides more benefit — an index on `phone` (unique, matches 1 document) or an index on `role` (matches ~60% for "passenger")?

**(b)** Create and test the phone index:

In [ ]:
%%mongodb
db.users.createIndex({"phone": 1})

In [ ]:
%%mongodb
db.users.find({"phone": "+6591234567"}).explain("executionStats")


**(c)** Create and test the role index:

In [ ]:
%%mongodb
db.users.createIndex({"role": 1})

In [ ]:
%%mongodb
db.users.find({"role": "passenger"}).explain("executionStats")


**(d)** Now test with a different value — `role: "merchant"` (~15% match):

In [ ]:
%%mongodb
db.users.find({"role": "merchant"}).explain("executionStats")


**(e)** Compare documents examined for `role: "passenger"` vs `role: "merchant"`. The same index has different selectivity depending on the value queried. This parallels Chapter 5's selectivity prediction on `customer_id` vs `status`.

---

## Part 6 — Comparison and Verdict

### Q15. Comparison paragraph

Write a short paragraph answering: which design would you pick for the super-app personalisation workload, and why? Reference specific queries from Q9–Q12 to support your argument. Name the workload characteristics (from the chapter's Algorithm step 1) that drive your choice.

Note: this is about the personalisation layer — the transactional core (ride matching, payments) is a separate question with a different answer.

---

### Q16. Bounded-growth rule — refined

State one explicit bounded-growth rule for your chosen design. Compare to your initial answer from Q2. Has anything changed after working through the queries and seeing the viral-merchant scenario from sanity check 3?

---

### Q17. Truth map

For every field that is duplicated (appears in more than one place), state: (a) where the editable truth lives, (b) where the snapshot/copy lives, (c) whether the copy is immutable (snapshot) or maintained (derived).

Example:

| Field | Editable truth | Snapshot location | Mutability |
|---|---|---|---|
| Plan name | `plans.name` | `users.subscriptionSnapshot.planNameAtSignUp` | Immutable (historical) |
| Commission rate | `plans.commissionRate` | `users.subscriptionSnapshot.commissionRateAtSignUp` | Immutable (contractual) |

Complete the table for all duplicated fields in your design.

---

## Deliverable

- **Part 1** (Q1–Q3): Hand-written JSON documents for Design A, bounded-growth rules, schema validation tests with rejection/acceptance evidence.
- **Part 2** (Q4–Q5): Design B collection inserts, reference resolution count, relational comparison sketch.
- **Part 3** (Q6–Q8): Dot notation query, `$elemMatch` counterexamples (passenger and merchant), regex with index discussion.
- **Part 4** (Q9–Q12): Cross-design queries with complexity comparison, cross-aggregate follow-up, update fan-out analysis with snapshot discussion.
- **Part 5** (Q13–Q14): `explain()` before/after with COLLSCAN → IXSCAN transition, selectivity prediction and verification.
- **Part 6** (Q15–Q17): Comparison paragraph, refined bounded-growth rule, truth map for all duplicated fields.

---

## Solutions

<details>
<summary><strong>Q1 Solution</strong></summary>

The `insertMany` statements are provided in the question. The key design points:

- Each role has a distinct document shape — no shared column definition.
- Passenger documents have `savedPlaces[]`, `paymentMethods[]`, `ridePreferences`. Sarah has 2 saved places; Wei Ming has 3; Rani has none (empty array — valid, no NULLs).
- Driver documents have `vehicle{}` (or `bike{}`), `license{}`, `earnings{}`, `preferences{}`, and `subscriptionSnapshot{}`.
- Merchant documents have `menu[]` with nested `customisations[]`, `operatingHours{}`, `promotions[]`.
- Plans are a separate collection because they are shared across users and updated independently.

The documents should look like the JSON the API returns — this is the JSON-first design principle.
</details>

---

<details>
<summary><strong>Q2 Solution</strong></summary>

| Array | Upper bound | Business rule | Fallback |
|---|---|---|---|
| `savedPlaces[]` | max 10 | App UI disables add-place at limit | User removes one before adding |
| `paymentMethods[]` | max 5 | App limits payment methods per user | User removes one before adding |
| `menu[]` | max 100 | Merchant portal enforces item limit | Move to `menuItems` collection, reference by merchantId |
| `menu[].customisations[]` | max 10 per item | Merchant portal enforces per-item limit | Unlikely to exceed; if so, paginate in UI |
| `promotions[]` | max 10 | Marketing limits concurrent promos per merchant | Archive expired promos |
| `preferences.preferredZones[]` | max 5 | Equal to total number of zones | Cannot exceed total zones |

Ride history is not embedded because it violates the **unbounded growth** condition — a daily commuter generates thousands of entries over years. It belongs in a separate `rideHistory` collection, referenced by `userId`.
</details>

---

<details>
<summary><strong>Q3 Solution</strong></summary>

**(b)** Rejected — the document lacks both `userId` and `role`, which are required fields.

**(c)** Rejected — `"admin"` is not in the allowed enum `["passenger", "driver", "merchant"]`.

**(d)** Accepted — the document has both required fields with valid values.

**(e)** The validation rule creates schema-on-write rigidity for the `role` field: adding a new role (e.g., `"corporate_admin"`) requires updating the validation rule. This demonstrates the trade-off: validation catches errors but reintroduces the migration friction that schema-on-read was designed to avoid. Production systems balance this by validating only critical fields (like `userId` and `role`) while leaving role-specific fields flexible.
</details>

---

<details>
<summary><strong>Q4 Solution</strong></summary>

The `insertMany` for `users_b` is provided. The remaining collections should contain:

- `preferences_b`: 7 documents (one per user with preferences — ride preferences for passengers, zone preferences for drivers, operating hours for merchants)
- `vehicles_b`: 2 documents (one per driver, containing vehicle/bike and licence)
- `earnings_b`: 2 documents (one per driver)
- `paymentMethods_b`: 4 documents (Sarah has 2, Wei Ming has 1, Rani has 1)
- `menuItems_b`: 9 documents (5 items for Ah Huat + 4 items for Tiger Sugar), each with `merchantId` field

Each document in the satellite collections must include `userId` (or `merchantId`) to link back to `users_b`. Note: Rani has no saved places, so no documents for her in `savedPlaces_b` — the absence of rows is how Design B represents an empty array.
</details>

---

<details>
<summary><strong>Q5 Solution</strong></summary>

**(a)** Design B requires 4 `$lookup` stages to assemble a complete driver profile: `vehicles_b` (vehicle + licence), `preferences_b` (zone preferences), `earnings_b` (earnings + incentive), `paymentMethods_b` (payment methods).

**(b)** Design A returns the complete nested document with one `findOne`. Design B requires a 5-stage pipeline (1 `$match` + 4 `$lookup`). Only Design A returns the nested JSON shape directly.

**(c)** Relational schema needs at minimum: `UserProfile`, `DriverProfile` (subtype), `PreferredZones` (junction table). The profile-load query requires 2–3 JOINs.

**(d)** Completed table:

| Design | Collections/Tables | Reads for profile load |
|---|---|---|
| Design A (embedded) | 1 collection | 1 `findOne` |
| Design B (referenced) | 5 collections | 1 `$match` + 4 `$lookup` |
| Relational | 3–4 tables | 2–3 JOINs |
</details>

---

<details>
<summary><strong>Q6 Solution</strong></summary>

Roughly one-third of drivers have expiry dates before 2026-01-01 (those with 2025 dates). With ~52 drivers, expect ~17 matches. The exact count depends on the seed data's random distribution.

The dot notation `"license.expiryDate"` navigates directly into the nested object — no JOIN, no subquery. String comparison works correctly for ISO-8601 dates because lexicographic order matches chronological order.
</details>

---

<details>
<summary><strong>Q7a Solution</strong></summary>

The without-`$elemMatch` count is higher because it matches documents where *any* place has frequency "daily" AND *any* place (possibly a different one) has `lastUsed` before 2025-01-15. A passenger whose Home is "daily" (lastUsed: 2025-01-20) and whose Parents place is "monthly" (lastUsed: 2024-12-25) would match without `$elemMatch` but not with it — no single place is both "daily" and old.

The with-`$elemMatch` count is lower because both conditions must be satisfied by the same array element.
</details>

---

<details>
<summary><strong>Q7b Solution</strong></summary>

Same principle as Q7a. A merchant with one "mains" item at 8.00 and one "drinks" item at 3.50 matches without `$elemMatch` (some item is "mains", some item is under 5.00) but not with `$elemMatch` (no single item is both "mains" and under 5.00).

In the relational model, each menu item is a separate row, so `WHERE category = 'mains' AND price < 5.00` naturally applies both conditions to the same row. Embedding collapses rows into an array, requiring `$elemMatch` to recover per-element filtering.
</details>

---

<details>
<summary><strong>Q8 Solution</strong></summary>

**(a)** All users match — every phone number in the dataset starts with `+65`.

**(b)** Returns "Ah Huat Chicken Rice" (and any seed-generated merchants whose business name contains "chicken rice").

**(c)** The `^\+65` pattern is index-friendly — the `^` anchor allows a B-tree index seek (like SQL's `LIKE '+65%'`). The `/chicken rice/i` pattern is not index-friendly — it requires scanning every value because the match can occur at any position. Prefer anchored patterns when the business question allows it.
</details>

---

<details>
<summary><strong>Q9 Solution</strong></summary>

Design A: 1 `findOne` returns the complete document. Design B: 5 pipeline stages (1 `$match` + 4 `$lookup`). Design A returns the nested JSON shape directly; Design B returns an assembled document with arrays-of-arrays from `$lookup` that need further processing.
</details>

---

<details>
<summary><strong>Q10 Solution</strong></summary>

Design A: 1 query with dot notation on the nested `license.expiryDate` field. Design B: 2 steps — query `vehicles_b` for expiring licences, then `$lookup` to `users_b` for driver identity (or vice versa). Design A is simpler because the licence data lives inside the same document as the driver identity.
</details>

---

<details>
<summary><strong>Q11 Solution</strong></summary>

Design A: 5 stages (`$match`, `$unwind`, `$group`, `$sort`, `$limit`). Requires `$unwind` because menu items are embedded arrays.

Design B: 3 stages (`$group`, `$sort`, `$limit`). No `$unwind` needed because each menu item is already a separate document in `menuItems_b`.

Design B is simpler for this particular analytics query — items are already "un-nested." But Design A is simpler for the dominant read (profile load). The design optimises for the dominant workload, not for every possible query.

**(e–f)** The cross-aggregate pipeline has 6 stages (adding `$lookup` and an extra `$unwind` for the plan). This demonstrates that crossing the aggregate boundary adds complexity — the same query in the relational model is just another JOIN.
</details>

---

<details>
<summary><strong>Q12 Solution</strong></summary>

**(c)** Business name is a field in the merchant's own document (Design A) or in `users_b` (Design B). Both designs require exactly 1 document update for this field.

**(d–e)** The update statement:

In [ ]:
%%mongodb
db.plans.updateOne(
    {"name": "GrabCar Premium"},
    {"$set": {"commissionRate": 0.18}}
)


Both designs update only the `plans` collection (1 document). Zero user profile documents are modified because:
- Design A uses immutable snapshots — `commissionRateAtSignUp` records the rate at sign-up and never changes.
- Design B uses references — `planId` always resolves to the current plan document.

**(f)** The difference is in read behaviour. Design A shows the driver the historical rate (0.20 — the rate they signed up under, stored in their snapshot). Design B shows the current rate (0.18 — resolved from the `plans` collection). For commission disputes, Design A is correct — the contractual rate should reflect the terms at sign-up, not the current rate. This is exactly the snapshot discipline from the chapter: label the copy, identify the truth, never update the snapshot.
</details>

---

<details>
<summary><strong>Q13 Solution</strong></summary>

**(b)** Before indexing: COLLSCAN, all ~207 documents examined.

**(e)** After indexing: IXSCAN, only driver documents with matching dates examined (~17 documents). The compound index `{"role": 1, "license.expiryDate": 1}` filters by `role` first (narrowing to ~52 drivers), then scans by date within that subset. This parallels Chapter 5's EXPLAIN exercise — the same before/after pattern, now on documents.
</details>

---

<details>
<summary><strong>Q14 Solution</strong></summary>

**(a)** The `phone` index provides dramatically more benefit — it matches exactly 1 document (high selectivity). The `role` index for "passenger" matches ~60% of documents (low selectivity).

**(c–d)** For `role: "passenger"`, the index examines ~123 documents (~60%). For `role: "merchant"`, the index examines ~32 documents (~15%). The same index has different selectivity depending on the value queried — selectivity is a property of the query, not just the index. This subtlety extends Chapter 5's binary "high vs low" comparison.
</details>

---

<details>
<summary><strong>Q15 Solution</strong></summary>

Design A (aggregate-first) is the better choice for the personalisation workload. Evidence:
- Q9: Profile load is 1 `findOne` vs 5 pipeline stages — the dominant read is dramatically simpler.
- Q10: Nested field queries use dot notation directly vs requiring `$lookup` to resolve references.
- Q12: Update complexity is similar for both designs for single-field changes.
- Q11: Analytics within the aggregate are slightly more complex (needing `$unwind`) but analytics across aggregate boundaries are expensive in both designs.

The workload characteristics that drive this choice: the dominant operation is an aggregate read (load complete profile on every session), structural variation is routine (different roles, tiers, features), and the personalisation layer does not require cross-document ACID transactions.
</details>

---

<details>
<summary><strong>Q16 Solution</strong></summary>

Example refined rule: "Embed `menu[]` up to 100 items; if a merchant exceeds this (the viral-merchant scenario), move to a separate `menuItems` collection referenced by `merchantId`. The merchant dashboard adds one `$lookup` stage, but the document stays within reasonable size. This fallback applies per-merchant — other merchants keep their embedded menus."

Compared to Q2: the initial rule may not have specified the fallback mechanism or considered what happens when the bound is exceeded. After seeing the viral-merchant scenario (sanity check 3) and comparing Design A vs Design B queries, the refined rule is more specific about the transition strategy.
</details>

---

<details>
<summary><strong>Q17 Solution</strong></summary>

| Field | Editable truth | Snapshot location | Mutability |
|---|---|---|---|
| Plan name | `plans.name` | `users.subscriptionSnapshot.planNameAtSignUp` | Immutable (historical — records plan name at sign-up) |
| Commission rate | `plans.commissionRate` | `users.subscriptionSnapshot.commissionRateAtSignUp` | Immutable (contractual — records rate agreed at sign-up) |
| Monthly price | `plans.monthlyPrice` | `users.subscriptionSnapshot.priceAtSignUp` | Immutable (historical — records price at sign-up) |

All snapshots are immutable — they record facts at a point in time. The editable truth lives in the `plans` collection. No field in this design is a "maintained" (derived) copy — all duplications are deliberate historical snapshots with the update rule "never update."
</details>